# Cum am construit `app/app.py` — tutorial Gradio (pas cu pas)

Ideea Gradio, într-o propoziție: **o funcție Python devine o interfață web.**
Tu scrii funcția, Gradio face caseta, butonul și layout-ul.

Construim app-ul incremental, exact în ordinea în care e scris `app.py`:
funcția simplă -> mai multe input-uri -> tab Chat -> starea partajată ->
regula subiect/știre -> tab Agent -> punem tab-urile împreună -> recapitulare.

Inspirat din [Gradio Quickstart](https://www.gradio.app/guides/quickstart).
Regula tutorialului: **cât mai simplu, doar esențialul.**

> `app/app.py` este doar un strat subțire de Gradio peste `core/` (agent, graph),
> construit în cursurile C2–C7. Aici nu rescriem `core/` — îl chemăm.
> Ca să ruleze fără chei API, folosim un backend fals.

In [1]:
%pip install -q gradio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

import gradio as gr
print("Gradio", gr.__version__)

d:\ADC 1\AI engineering\echochamber-project-team3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gradio 6.14.0


## 1. Cel mai simplu Gradio

`gr.Interface` are nevoie de 3 lucruri: `fn` (funcția), `inputs`, `outputs`.

In [3]:
def saluta(nume):
    return "Salut, " + nume

gr.Interface(fn=saluta, inputs="text", outputs="text").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Atât. Caseta, butonul *Submit*, totul l-a făcut Gradio. Pe asta se construiește
toată aplicația.

## 2. Mai multe input-uri = o listă

Tab-urile noastre au mai multe câmpuri. Dacă funcția are mai multe argumente,
dai o **listă** la `inputs` (ordinea = ordinea argumentelor).

In [4]:
def combina(text, optiune, numar):
    return f"[{optiune} @ {numar}] {text}"

gr.Interface(
    fn=combina,
    inputs=[gr.Textbox(label="Text"),
            gr.Dropdown(["a", "b"], value="a", label="Opțiune"),
            gr.Slider(0, 1, value=0.3, step=0.1, label="Număr")],
    outputs=gr.Textbox(label="Rezultat"),
).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Reține tiparul `[text, dropdown, slider] -> funcție -> text`. **Asta e un tab.**

## 3. Backend fals (ca să rulăm fără chei API)

În `app.py` real, sus, sunt 2 importuri din `core/` (construite în C6–C7):

```python
from core.agent import generate_agent_response   # un agent RAG
from core.graph import run_thread                 # dezbatere multi-agent
```

Aici le înlocuim cu funcții-jucărie. Restul codului rămâne identic ca structură.

In [5]:
def fake_llm(prompt):
    return "(răspuns simulat) despre: " + prompt[:70]

def fake_agent(slug, stimulus):
    voci = {"anti_sistem": "Instituțiile par din nou rupte de oameni.",
            "pro_european": "Să discutăm pe baza procedurilor."}
    return voci.get(slug, f"[{slug}] {stimulus[:50]}")

AGENTS = [("Anti-sistem", "anti_sistem"), ("Pro-european", "pro_european")]
print("backend fals pregătit")

backend fals pregătit


## 4. Primul tab real: Chat

În `app.py`, tab-ul Chat e funcția `chat()` + un `gr.Interface`. O reproducem
cu `fake_llm`.

In [6]:
def chat(prompt):
    return fake_llm(prompt) if prompt.strip() else "Scrie un prompt."

gr.Interface(
    fn=chat,
    inputs=gr.Textbox(label="Întrebare / prompt", lines=4),
    outputs=gr.Textbox(label="Răspuns", lines=10),
    title="Chat",
).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Tab-ul Chat complet, fără să scriem vreun buton. Gradio l-a făcut.

## 5. Starea partajată: `CFG` și `ART`

Tab-ul **Setări** alege modelul și (opțional) încarcă o știre. Celelalte tab-uri
trebuie să **vadă** acea știre. Soluția minimă: două dicționare la nivel de modul.

- `CFG` = provider / model / temperatură
- `ART` = textul + titlul știrii încărcate

Setări **scrie** în ele, restul tab-urilor **citesc**. (Alternativa „canonică"
ar fi `gr.State`; varianta minimă alege simplitatea.)

Truc din `app.py`: provider + model sunt **un singur dropdown**
(`"provider|model"`) — imposibil să fie nepotrivite.

In [7]:
CFG = {"provider": "gemini", "model": "gemini-2.5-flash-lite", "temp": 0.3}
ART = {"text": "", "title": ""}

MODEL_CHOICES = [("gemini · gemini-2.5-flash-lite", "gemini|gemini-2.5-flash-lite"),
                 ("deepseek · deepseek-chat",       "deepseek|deepseek-chat")]

def setup(model_choice, temperature, fake_url):
    provider, model = model_choice.split("|", 1)     # despărțim "provider|model"
    CFG.update(provider=provider, model=model, temp=temperature)
    if fake_url.strip():
        ART.update(text=f"Text fals al știrii de la {fake_url}", title=fake_url)
        return f"Setări salvate. Știre ACTIVĂ: {fake_url}"
    ART.update(text="", title="")
    return f"Setări salvate ({provider} · {model}). Fără știre."

gr.Interface(
    fn=setup,
    inputs=[gr.Dropdown(MODEL_CHOICES, value=MODEL_CHOICES[0][1],
                        label="Provider · Model"),
            gr.Slider(0, 1, value=0.3, step=0.1, label="Temperatură"),
            gr.Textbox(label="URL știre (gol = fără știre)")],
    outputs=gr.Textbox(label="Stare"),
    title="Setări",
).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## 6. Regula cheie: subiectul intră *peste* știre

`_subject()` decide ce primește un agent:

- **știre + subiect** -> vorbim despre subiect, dar **în contextul știrii**
- **doar știre** -> vorbim despre știre
- **doar subiect** -> vorbim doar despre subiect

In [8]:
def _subject(typed):
    typed = (typed or "").strip()
    news = ART["text"].strip()
    if news and typed:
        return f"{typed}\n\n[În contextul acestei știri:]\n{news[:600]}"
    if news:
        return news[:700]
    return typed

ART.update(text="Știre despre UE și energie.")
print(_subject("Bolojan"))     # subiect peste știre
ART.update(text="")
print(_subject("Bolojan"))     # doar subiect

Bolojan

[În contextul acestei știri:]
Știre despre UE și energie.
Bolojan


## 7. Tab-ul Agent

Tab-ul Agent = `_subject()` + chemarea backend-ului. În `app.py` real,
`fake_agent` e `generate_agent_response` din `core.agent` (C6).

In [9]:
def agent(text, slug):
    s = _subject(text)
    if not s.strip():
        return "Încarcă o știre sau scrie un subiect."
    return fake_agent(slug, s)               # în app: generate_agent_response(...)

gr.Interface(
    fn=agent,
    inputs=[gr.Textbox(label="Subiect (intră peste știre, dacă e încărcată)",
                       lines=3),
            gr.Dropdown(AGENTS, value="anti_sistem", label="Agent")],
    outputs=gr.Textbox(label="Comentariu", lines=10),
    title="Agent",
).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Tab-urile **Rezumat**, **Toți agenții** și **Dezbatere** au exact același tipar:
o funcție + un `gr.Interface`. Doar funcția diferă (rezumat / loop pe roluri /
`core.graph.run_thread`).

## 8. Punem tab-urile împreună

`app.py` are 6 tab-uri cu o **temă comună**. `gr.TabbedInterface` nu acceptă
`theme=` pe toate versiunile, așa că facem ce face el intern: un `gr.Blocks`
cu temă, `gr.Tabs`, și randăm fiecare `Interface` cu `.render()`.

In [10]:
tab_setup = gr.Interface(setup,
    [gr.Dropdown(MODEL_CHOICES, value=MODEL_CHOICES[0][1], label="Provider · Model"),
     gr.Slider(0, 1, value=0.3, step=0.1, label="Temperatură"),
     gr.Textbox(label="URL știre")],
    gr.Textbox(label="Stare"), title="Setări")

tab_chat = gr.Interface(chat, gr.Textbox(label="Prompt", lines=3),
    gr.Textbox(label="Răspuns", lines=8), title="Chat")

tab_agent = gr.Interface(agent,
    [gr.Textbox(label="Subiect", lines=3),
     gr.Dropdown(AGENTS, value="anti_sistem", label="Agent")],
    gr.Textbox(label="Comentariu", lines=8), title="Agent")

TABS = [("Setări", tab_setup), ("Chat", tab_chat), ("Agent", tab_agent)]

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown("# EchoChamber Studio")
    with gr.Tabs():
        for nume, iface in TABS:
            with gr.Tab(nume):
                iface.render()

demo.launch()

C:\Users\Kirby\AppData\Local\Temp\ipykernel_8320\3885253093.py:17: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Acesta e scheletul exact din `app.py`. În aplicația reală sunt 6 tab-uri în
loc de 3, iar funcțiile cheamă `core/` în loc de `fake_*`.

## 9. Recapitulare

**Ce face Gradio (tot tutorialul, pe scurt):**

1. `gr.Interface(fn, inputs, outputs)` — o funcție devine interfață.
2. Mai multe input-uri = o listă. Un astfel de bloc = un tab.
3. `CFG` / `ART` (dict-uri de modul) = starea partajată: Setări scrie, restul citesc.
4. `_subject()` = regula subiect-peste-știre.
5. `gr.Blocks` + `gr.Tabs` + `.render()` = cele 6 tab-uri cu temă comună.

**Dependențe (din structura repo):** `app/app.py` cheamă doar `core/` —
nu rescrie nimic.

| Tab(uri) | Funcție în app.py | Backend | Curs |
|---|---|---|---|
| Setări · Chat · Rezumat | `setup` · `chat` · `summary` | apel LLM direct | C2 |
| Agent | `agent` -> `_agent` | `core.agent` (FAISS + rol) | C5 + C6 |
| Toți agenții | `all_agents` | loop pe `roles.yaml` -> `core.agent` | C6 |
| Dezbatere | `debate` | `core.graph.run_thread` (LangGraph) | C7 |

`core.agent` -> `core.retriever` (FAISS) + `roles.yaml` + LLM.
`core.graph` orchestrează `core.agent` (round-robin). Singura punte offline->runtime:
vectorstore-urile construite offline, citite de retriever la fiecare cerere.

**Mesajul cheie:** aplicația nu e un proiect nou. E un strat subțire Gradio
peste funcțiile din C2–C7. Fiecare tab = un buton peste o funcție de curs.

## Tema 3 — Extensii individuale (student_04)

Am adăugat 3 modificări vizibile față de aplicația de bază din tutorial:
1. **Tab nou** — „Despre / Etică"
2. **Funcție nouă** — numărare cuvinte și curățare text
3. **Modificare temă și design** — temă diferită, titlu mai clar, emoji

In [12]:
import gradio as gr

# ── Backend fals (fără chei API) ──────────────────────────────────────────────

def fake_llm(prompt):
    return "(răspuns simulat) despre: " + prompt[:70]

def fake_agent(slug, stimulus):
    voci = {
        "anti_sistem":   "Instituțiile par din nou rupte de oameni.",
        "pro_european":  "Să discutăm pe baza procedurilor și a dovezilor.",
        "conspirationist": "Cine trage sforile? Cui îi folosește această decizie?"
    }
    return voci.get(slug, f"[{slug}] {stimulus[:50]}")

AGENTS = [
    ("Anti-sistem",     "anti_sistem"),
    ("Pro-european",    "pro_european"),
    ("Conspiraționist", "conspirationist"),
]

# ── Funcție nouă: curățare și numărare cuvinte ────────────────────────────────

def analizeaza_text(text):
    """Curăță textul și returnează statistici simple."""
    if not text.strip():
        return "Scrie un text mai întâi.", ""
    
    text_curatat = text.strip()
    cuvinte = len(text_curatat.split())
    caractere = len(text_curatat)
    propozitii = text_curatat.count('.') + text_curatat.count('!') + text_curatat.count('?')
    
    stats = (
        f"📝 Cuvinte: {cuvinte}\n"
        f"🔤 Caractere: {caractere}\n"
        f"📌 Propoziții estimate: {max(propozitii, 1)}"
    )
    return text_curatat, stats

# ── Funcție chat ──────────────────────────────────────────────────────────────

def chat(prompt):
    return fake_llm(prompt) if prompt.strip() else "Scrie un prompt."

# ── Funcție agent ─────────────────────────────────────────────────────────────

def agent(text, slug):
    if not text.strip():
        return "Scrie un subiect sau încarcă o știre."
    return fake_agent(slug, text)

# ── Tab: Despre / Etică ───────────────────────────────────────────────────────

despre_text = """
## 🧠 Despre EchoChamber

**EchoChamber** este un instrument de simulare și analiză a discursului politic românesc.

### Ce face aplicația?
- Simulează răspunsuri ale unor agenți discursivi cu roluri politice diferite
- Folosește RAG (Retrieval-Augmented Generation) pentru a recupera context din corpus
- Organizează conversații multi-agent cu LangGraph

### ⚠️ Limite etice
| Risc | Explicație |
|---|---|
| Antropomorfizare | Agenții nu sunt persoane reale |
| Halucinație | Modelul poate genera afirmații nesusținute |
| Amplificare discursivă | Interacțiunea poate intensifica conflictul |
| Bias din corpus | Corpusul poate conține limbaj problematic |

### Disclaimer
EchoChamber este un prototip educațional și de cercetare. 
Agenții sunt constructe analitice fictive, nu reprezentanți ai unor grupuri reale. 
Outputurile nu trebuie prezentate ca fapte sau opinii reale ale unor comunități.

---
*Proiect realizat în cadrul cursului AI Engineering
"""

def show_despre():
    return despre_text

# ── Construim aplicația cu gr.Blocks ─────────────────────────────────────────

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="violet", neutral_hue="slate"),
    title="EchoChamber Studio — student_04"
) as demo_tema3:

    gr.Markdown("# 🗣️ EchoChamber Studio")
    gr.Markdown("*Simulare de discurs politic românesc — extensie Tema 3, student_04*")
    gr.Markdown("---")

    with gr.Tab("💬 Chat"):
        gr.Markdown("### Trimite un prompt simplu la model")
        prompt_box = gr.Textbox(
            label="Prompt",
            value="Explică în 2 propoziții ce este un LLM.",
            lines=4
        )
        chat_btn = gr.Button("▶ Trimite", variant="primary")
        chat_out = gr.Textbox(label="Răspuns", lines=8)
        chat_btn.click(fn=chat, inputs=prompt_box, outputs=chat_out)

    with gr.Tab("🤖 Agent"):
        gr.Markdown("### Generează un răspuns din perspectiva unui agent")
        text_box = gr.Textbox(
            label="Subiect / stimulus politic",
            value="CCR a decis anularea alegerilor după suspiciuni privind influențe externe.",
            lines=4
        )
        agent_drop = gr.Dropdown(
            choices=AGENTS,
            value="anti_sistem",
            label="Agent"
        )
        agent_btn = gr.Button("▶ Generează", variant="primary")
        agent_out = gr.Textbox(label="Comentariu agent", lines=8)
        agent_btn.click(fn=agent, inputs=[text_box, agent_drop], outputs=agent_out)

    with gr.Tab("📊 Analizează text"):
        gr.Markdown("### Curăță și analizează un text politic")
        input_text = gr.Textbox(
            label="Text de analizat",
            value="CCR a decis anularea alegerilor după suspiciuni privind influențe externe.",
            lines=5
        )
        analiza_btn = gr.Button("▶ Analizează", variant="primary")
        text_curatat_out = gr.Textbox(label="Text curățat", lines=5)
        stats_out = gr.Textbox(label="Statistici", lines=4)
        analiza_btn.click(
            fn=analizeaza_text,
            inputs=input_text,
            outputs=[text_curatat_out, stats_out]
        )

    with gr.Tab("ℹ️ Despre / Etică"):
        gr.Markdown(despre_text)

demo_tema3.launch()

C:\Users\Kirby\AppData\Local\Temp\ipykernel_8320\2857691804.py:87: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


## Explicație Tema 3

**Ce am adăugat:**
Am extins aplicația din tutorial cu 3 modificări vizibile față de versiunea de bază.

**Tab nou creat:**
„Despre / Etică" — conține un disclaimer, limitele etice ale simulării și o scurtă descriere a proiectului, organizate ca tabel Markdown.

**Funcție nouă:**
`analizeaza_text()` — primește un text politic, îl curăță și returnează statistici simple: număr de cuvinte, caractere și propoziții estimate. Tab-ul „Analizează text" expune această funcție în interfață.

**Modificare de design și temă:**
Am schimbat tema la `gr.themes.Soft(primary_hue="violet")`, am adăugat emoji în titlurile tab-urilor, un subtitle și un separator vizual (`---`) pentru a structura mai clar interfața.

**Ce aș îmbunătăți:**
Funcția `analizeaza_text()` ar putea detecta automat limbajul emoțional sau cuvintele-cheie politice frecvente din corpus, nu doar statistici de bază.